In [ ]:
import pandas
import spacy
import re

nlp = spacy.load('en_core_web_lg')

In [ ]:
df = pandas.read_csv('reddit_depression_dataset.csv')
df

In [ ]:
df = df[['body','label']]
df

In [ ]:
df = df.dropna()
df = df.drop_duplicates()
df

In [ ]:
sample 1000 rows from label 0 and 1
sample_df = pandas.concat([df[df['label'] == 0].sample(n=1000, random_state=42),df[df['label'] == 1].sample(n=1000, random_state=42)])
sample_df

In [ ]:
import nltk
from nltk.corpus import stopwords
from textblob import Word

nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
stop = stopwords.words('english')

def remove_noise(text):
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.replace('\t', ' ')
    text = text.lower()
    text = re.sub(r'#\w+', '', text) # Remove hashtags
    text = re.sub(r'@\w+', '', text) # Remove mentions
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Remove URLs
    text = ' '.join([word for word in text.split() if word.lower() not in stop])
    return text

In [ ]:
sample_df['body'] = sample_df['body'].apply(remove_noise)

In [ ]:
# lemmatization
sample_df['body']= sample_df['body'].apply(lambda x: ' '.join([Word(word).lemmatize() for word in x.split()]))

In [ ]:
# GloVe embeddings
# def glove_embedding(text):
#     doc = nlp(text)
#     return doc.vector

# sample_df['embedding'] = sample_df['body'].apply(glove_embedding)
# To long to take

In [ ]:
# Import required libraries for CNN-LSTM
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

In [ ]:
# Prepare data for CNN-LSTM
text_lengths =[len(text.split()) for text in sample_df['body']]
max_len = int(np.percentile(text_lengths, 95))  # Use 95th percentile for max length

all_word = set()
for text in sample_df['body']:
    for word in text.split():
        all_word.add(word)
len_bow = len(all_word)

max_words = int(len_bow*0.95)

# Use the cleaned text data
texts = sample_df['body'].values
labels = sample_df['label'].values

# Tokenize texts
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Testing labels shape: {y_test.shape}")

# Build CNN-LSTM model
def create_cnn_lstm_model():
    # Input layer
    input_layer = Input(shape=(max_len,))
    
    # Embedding layer
    embedding = Embedding(max_words, 128, input_length=max_len)(input_layer)
    
    # CNN layers for local feature extraction
    conv1 = Conv1D(64, 3, activation='relu')(embedding)
    pool1 = MaxPooling1D(2)(conv1)
    
    conv2 = Conv1D(32, 3, activation='relu')(pool1)
    pool2 = MaxPooling1D(2)(conv2)
    
    # LSTM layer for sequence modeling
    lstm_out = LSTM(50, dropout=0.2, recurrent_dropout=0.2)(pool2)
    
    # Dense layers
    dense1 = Dense(50, activation='relu')(lstm_out)
    dropout1 = Dropout(0.5)(dense1)
    
    # Output layer
    output = Dense(1, activation='sigmoid')(dropout1)
    
    model = Model(inputs=input_layer, outputs=output)
    return model

# Create and compile model
model = create_cnn_lstm_model()
model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

# Display model summary
model.summary()

# Train the model
print("Training CNN-LSTM model...")
history = model.fit(X_train, y_train,
                    batch_size=32,
                    epochs=10,
                    validation_data=(X_test, y_test),
                    verbose=1)


In [ ]:

# Make predictions
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)


In [ ]:
print(max_len,max_words)

In [ ]:
comb_accuracy = accuracy_score(y_test, y_pred)
comb_roc_auc = roc_auc_score(y_test, y_pred_proba)

# Evaluate CNN-LSTM model
print("\n=== CNN-LSTM Model Results ===")
print(classification_report(y_test, y_pred))
print("Accuracy:", comb_accuracy)
print("ROC AUC Score:", comb_roc_auc)

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

# Plot ROC curve for CNN-LSTM
plt.figure(figsize=(8, 6))
fpr_cnn, tpr_cnn, _ = roc_curve(y_test, y_pred_proba)
plt.plot(fpr_cnn, tpr_cnn, label='CNN-LSTM ROC curve (area = {:.2f})'.format(roc_auc_score(y_test, y_pred_proba)))
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - CNN-LSTM Model')
plt.legend(loc='lower right')
plt.show()


In [ ]:
# TF-IDF with RandomForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(sample_df['body'])
# Transform to DataFrame
tfidf_df = pandas.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
tfidf_df.index = sample_df.index
sample_df = pandas.concat([tfidf_df, sample_df['label']], axis=1)

# Split data into features and labels
X = sample_df.drop('label', axis=1)
y = sample_df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Create and train RandomForest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


In [ ]:
# Make predictions
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluate RandomForest model
tfidf_rm_accuracy = accuracy_score(y_test, y_pred_rf)
tfidf_rm_roc_auc = roc_auc_score(y_test, y_pred_proba_rf)
print("\n=== RandomForest Model Results ===")
print(classification_report(y_test, y_pred_rf))
print("Accuracy:", tfidf_rm_accuracy)
print("ROC AUC Score:", tfidf_rm_roc_auc)

# Plot ROC curve for RandomForest
plt.figure(figsize=(8, 6))
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
plt.plot(fpr_rf, tpr_rf, label='RandomForest ROC curve (area = {:.2f})'.format(tfidf_rm_roc_auc))
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - RandomForest Model')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# comparison
print("\n=== Model Comparison ===")
print(f"CNN-LSTM Model - Accuracy: {comb_accuracy:.4f}, ROC AUC: {comb_roc_auc:.4f}")
print(f"RandomForest Model - Accuracy: {tfidf_rm_accuracy:.4f}, ROC AUC: {tfidf_rm_roc_auc:.4f}")